# Data Encryption in Backblaze B2

This notebook guides you through the different options for encrypting data in Backblaze B2, with Python examples that use the AWS SDK for Python (boto3).

First, you will need to install boto3 and pycryptodome.

In [1]:
%pip install boto3 pycryptodome


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Restart your Jupyter kernel so that it can use the installed modules.

Define your bucket and file names. You must change these to suit your environment!

In [2]:
# Bucket with default encryption disabled
DEMO_BUCKET_NAME = 'metadaddy-demo'
# Bucket with default encryption enabled
DEFAULT_SSE_B2_BUCKET_NAME = 'metadaddy-sse-b2'
# A local file that we will upload
FILENAME = 'photo.png'

## Create a boto3 Client

This example uses a profile in the AWS common configuration file.

To create a profile for Backblaze B2, run `aws configure --profile b2` and enter your application key ID and application key at the `AWS Access Key ID` and `AWS Secret Access Key` prompts respectively, and the region component of your endpoint, for example, `us-west-004` for the `Default region name`. You must then edit `~/.aws/config` on macOS or Linux, or `C:\Users\USERNAME\.aws\config` on Windows, and add your endpoint URL to the new profile, like this:

```
[profile b2]
region = us-west-004
endpoint_url = https://s3.us-west-004.backblazeb2.com
```

Alternatively, uou can use any supported mechanism for configuring boto3, such as setting environment variables (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` and `AWS_ENDPOINT_URL`), or passing credentials directly to boto3.client().

Note that you MUST set the endpoint URL, no matter which configuration mechanism you use!

In [3]:
import boto3
from botocore.exceptions import ClientError

boto3.setup_default_session(profile_name='b2')

client = boto3.client('s3')

## Writing Data to Backblaze B2 With No Encryption

First, we'll try to get encryption settings for a bucket with default encryption disabled.

Note that you have to catch the `ClientError` that is thrown if the bucket does not have encryption set by default.

In [4]:
try:
    response = client.get_bucket_encryption(
        Bucket=DEMO_BUCKET_NAME
    )
except ClientError as e:
    print(f'ClientError: {e}')

ClientError: An error occurred (ServerSideEncryptionConfigurationNotFoundError) when calling the GetBucketEncryption operation: The server side encryption configuration was not found


Now we'll read a local file into memory. We'll upload this data with different encryption options throughout the remainder of the notebook.

In [5]:
with open(FILENAME, 'rb') as f:
    # Read the entire file into memory
    plaintext_data = f.read()

    print(f'Read {len(plaintext_data)} bytes')

Read 943923 bytes


Writing data to the bucket with default encryption disabled, and no encryption settings in the `put_object` call, will write data with no encryption.

In [6]:
response = client.put_object(
    Bucket=DEMO_BUCKET_NAME,
    Key=FILENAME,
    Body=plaintext_data,
)

We can pretty-print the full response to see that there are no encryption-related headers.

In [7]:
from pprint import pprint

pprint(response)

{'ChecksumCRC32': 'HW87VQ==',
 'ChecksumType': 'FULL_OBJECT',
 'ETag': '"79320ebd9d856f093084ddda533a695e"',
 'ResponseMetadata': {'HTTPHeaders': {'cache-control': 'max-age=0, no-cache, '
                                                       'no-store',
                                      'connection': 'keep-alive',
                                      'content-length': '0',
                                      'date': 'Wed, 15 Oct 2025 16:47:44 GMT',
                                      'etag': '"79320ebd9d856f093084ddda533a695e"',
                                      'server': 'nginx',
                                      'strict-transport-security': 'max-age=63072000',
                                      'x-amz-checksum-crc32': 'HW87VQ==',
                                      'x-amz-checksum-type': 'FULL_OBJECT',
                                      'x-amz-id-2': 'aMTc1FmaxOTUzFzUVYz5mUjQZZABjgmL4',
                                      'x-amz-request-id': '0a1efdf7f74d5

Let's define a simple function to show just the encryption-related HTTP headers from the response.

In [8]:
def print_encryption_headers(response):
    # This dict comprehension selects any headers that contain the substring 'encryption' in the key
    pprint({key: value for key, value in response['ResponseMetadata']['HTTPHeaders'].items() if 'encryption' in key})

 Now it's a little easier to see that there are no encryption-related headers in the `put_object` response:

In [9]:
print_encryption_headers(response)

{}


## Writing Data to Backblaze B2 with SSE-B2

If we want to use SSE-B2 with a bucket that does not have default encryption enabled, we need to specify the `ServerSideEncryption` argument.

In [10]:
response = client.put_object(
    Bucket=DEMO_BUCKET_NAME,
    Key=f'sse-b2-{FILENAME}',
    Body=plaintext_data,
    ServerSideEncryption='AES256'
)

Note the server-side encryption header in the response

In [11]:
print_encryption_headers(response)

{'x-amz-server-side-encryption': 'AES256'}


When we read the data back, we don't need to specify any extra parameters

In [12]:
response = client.get_object(
    Bucket=DEMO_BUCKET_NAME,
    Key=f'sse-b2-{FILENAME}',
)

Note the same server side encryption header in the `get_object` response:

In [13]:
print_encryption_headers(response)

{'x-amz-server-side-encryption': 'AES256'}


Compare the downloaded data to the plaintext we uploaded. No output means that the data is identical.

In [14]:
downloaded_data = response['Body'].read()
assert downloaded_data == plaintext_data

Now let's get encryption settings for the bucket with default encryption enabled

In [15]:
response = client.get_bucket_encryption(
    Bucket=DEFAULT_SSE_B2_BUCKET_NAME
)
pprint(response['ServerSideEncryptionConfiguration'])

{'Rules': [{'ApplyServerSideEncryptionByDefault': {'SSEAlgorithm': 'AES256'}}]}


Since this bucket has default encryption enabled, we don't need to specify any extra parameters to have Backblaze B2 encrypt the data.

Again, note the server-side encryption header in the response.

In [16]:
response = client.put_object(
    Bucket=DEFAULT_SSE_B2_BUCKET_NAME,
    Key=f'sse-b2-default-{FILENAME}',
    Body=plaintext_data
)
print_encryption_headers(response)

{'x-amz-server-side-encryption': 'AES256'}


Read the newly uploaded object

In [17]:
response = client.get_object(
    Bucket=DEFAULT_SSE_B2_BUCKET_NAME,
    Key=f'sse-b2-default-{FILENAME}',
)
print_encryption_headers(response)

{'x-amz-server-side-encryption': 'AES256'}


Compare the downloaded data just as before

In [18]:
downloaded_data = response['Body'].read()
assert downloaded_data == plaintext_data

## Writing Data to Backblaze B2 with SSE-C

SSE-C uses AES-256, so we need to create a 256-bit (32-byte) key.

In [19]:
from Crypto.Random import get_random_bytes

sse_c_key = get_random_bytes(32)

Here we just save the key to a file on disk. You would not do this in a real production system!

In [20]:
with open('sse_c_keyfile', 'wb') as f:
    f.write(sse_c_key)

Now we can write the file using SSE-C. We must include the key in the `put_object` call as well as the `SSECustomerAlgorithm` argument.

In [21]:
response = client.put_object(
    Bucket=DEMO_BUCKET_NAME,
    Key=f'sse-c-{FILENAME}',
    Body=plaintext_data,
    SSECustomerAlgorithm='AES256',
    SSECustomerKey=sse_c_key
    # No need to provide SSECustomerKeyMD5 - the SDK calculates it
)

Again, the response headers confirm the encryption mechanism. We can also see the MD5 hash of the encryption key.

In [22]:
print_encryption_headers(response)

{'x-amz-server-side-encryption-customer-algorithm': 'AES256',
 'x-amz-server-side-encryption-customer-key-md5': 'sOYkCwTRVrn32PXe9P6ZCA=='}


What happens if we try to read the file without the key?

In [23]:
try:
    response = client.get_object(
        Bucket=DEMO_BUCKET_NAME,
        Key=f'sse-c-{FILENAME}'
    )
except Exception as e:
    print(f'Exception: {e}')

Exception: a bytes-like object is required, not 'StreamingChecksumBody'


What happens if we try to read the file with the wrong key?

In [24]:
bad_key = get_random_bytes(32)
try:
    response = client.get_object(
        Bucket=DEMO_BUCKET_NAME,
        Key=f'sse-c-{FILENAME}',
        SSECustomerAlgorithm='AES256',
        SSECustomerKey=bad_key
    )
except Exception as e:
    print(f'Exception: {e}')

Exception: a bytes-like object is required, not 'StreamingChecksumBody'


Let's read the file with the correct key

In [25]:
response = client.get_object(
    Bucket=DEMO_BUCKET_NAME,
    Key=f'sse-c-{FILENAME}',
    SSECustomerAlgorithm='AES256',
    SSECustomerKey=sse_c_key
)
print_encryption_headers(response)

{'x-amz-server-side-encryption-customer-algorithm': 'AES256',
 'x-amz-server-side-encryption-customer-key-md5': 'sOYkCwTRVrn32PXe9P6ZCA=='}


Again, the downloaded data should be identical to what we uploaded.

In [26]:
downloaded_data = response['Body'].read()
assert downloaded_data == plaintext_data

## Client-side encryption using PyCryptodome

We must create a new key - we shouldn't use the one we already shared with B2!

In [27]:
cse_key = get_random_bytes(32)

with open('cse_keyfile', 'wb') as f:
    f.write(cse_key)

For this example, we'll use AES Galois/Counter Mode (GCM) to encrypt data at the client. First, we create a new cipher object.

In [28]:
from Crypto.Cipher import AES

cipher = AES.new(cse_key, AES.MODE_GCM)

Now we can encrypt the data. GCM provides data integrity by generating an authentication tag during encryption that verifies the data has not been tampered with.

In [29]:
ciphertext, tag = cipher.encrypt_and_digest(plaintext_data)

We can verify that the ciphertext is at least different from the plaintext!

In [30]:
assert ciphertext != plaintext_data

The nonce is a random value generated when we created the cipher. It adds randomness to the process, ensuring that encrypting the same plaintext will result in a different ciphertext.

In [31]:
nonce = cipher.nonce

How big are these cryptographic objects?

In [32]:
print(f"The plaintext data is {len(plaintext_data)} bytes")
print(f"The nonce is {len(nonce)} bytes; the ciphertext is {len(ciphertext)} bytes, and the tag is {len(tag)} bytes")

The plaintext data is 943923 bytes
The nonce is 16 bytes; the ciphertext is 943923 bytes, and the tag is 16 bytes


As you can see, the nonce and tag are 16 bytes, while the ciphertext is the same length as the plaintext data.

The convention is to concatenate the nonce, ciphertext and tag to form the encrypted data. The nonce is required to decrypt the ciphertext, and the tag allows us to verify that the data has not been tampered with in transit.

In [33]:
encrypted_data = nonce + ciphertext + tag

Now we can upload the encrypted data. Note that we need not specify any encryption-related arguments in the `put_object` call. As far as Backblaze B2 is concerned, this is just data.

In [34]:
response = client.put_object(
    Bucket=DEMO_BUCKET_NAME,
    Key=f'encrypted-{FILENAME}',
    Body=encrypted_data,
)
print_encryption_headers(response)

{}


Download the encrypted data. Again, no special arguments are required.

In [35]:
response = client.get_object(
    Bucket=DEMO_BUCKET_NAME,
    Key=f'encrypted-{FILENAME}',
)

Extract the nonce, ciphertext and tag from the downloaded data

In [36]:
downloaded_data = response['Body'].read()

# Python list slicing allows us to easily extract the different components from the downloaded data
downloaded_nonce = downloaded_data[:16]
downloaded_ciphertext = downloaded_data[16:-16]
downloaded_tag = downloaded_data[-16:]

Now we can decrypt the data and verify that it matches the plaintext

In [37]:
cipher = AES.new(cse_key, AES.MODE_GCM, nonce=downloaded_nonce)

decrypted_data = cipher.decrypt_and_verify(downloaded_ciphertext, downloaded_tag)

assert decrypted_data == plaintext_data

## Encrypting Existing Data in Backblaze B2

To change the encryption mechanism for existing data to SSE-B2 or SSE-C, you can perform a server-side copy and provide the desired encryption settings.

First, we'll replace the unencrypted file with an SSE-B2 encrypted copy.

In [38]:
response = client.copy_object(
    CopySource={
        'Bucket': DEMO_BUCKET_NAME,
        'Key': FILENAME
    },
    # Destination
    Bucket=DEMO_BUCKET_NAME,
    Key=FILENAME,
    ServerSideEncryption='AES256'
)
print_encryption_headers(response)

{'x-amz-server-side-encryption': 'AES256'}


Remember, B2 buckets are versioned, so we need to remember to delete the original unencrypted file!

The response from `copy_object` includes the version ID of the original file, so it's straightforward to delete that specific version. The 204 response indicates success.

In [39]:
response = client.delete_object(
    Bucket=DEMO_BUCKET_NAME,
    Key=FILENAME,
    VersionId=response['CopySourceVersionId']
)
pprint(response['ResponseMetadata']['HTTPStatusCode'])

204


To replace the SSE-B2-encrypted file with SSE-C we need to provide the SSE-C key, as before.

In [40]:
response = client.copy_object(
    CopySource={
        'Bucket': DEMO_BUCKET_NAME,
        'Key': FILENAME
    },
    # Destination
    Bucket=DEMO_BUCKET_NAME,
    Key=FILENAME,
    SSECustomerAlgorithm='AES256',
    SSECustomerKey=sse_c_key
    # No need to provide SSECustomerKeyMD5 - the SDK calculates it
)
print_encryption_headers(response)

{'x-amz-server-side-encryption-customer-algorithm': 'AES256',
 'x-amz-server-side-encryption-customer-key-md5': 'sOYkCwTRVrn32PXe9P6ZCA=='}


Again, delete the original!

In [41]:
response = client.delete_object(
    Bucket=DEMO_BUCKET_NAME,
    Key=FILENAME,
    VersionId=response['CopySourceVersionId']
)
pprint(response['ResponseMetadata']['HTTPStatusCode'])

204


To encrypt existing data with CSE, first we need to GET the plaintext

In [42]:
source = client.get_object(
    Bucket=DEMO_BUCKET_NAME,
    Key=FILENAME,
    SSECustomerAlgorithm='AES256',
    SSECustomerKey=sse_c_key
)
print_encryption_headers(source)

{'x-amz-server-side-encryption-customer-algorithm': 'AES256',
 'x-amz-server-side-encryption-customer-key-md5': 'sOYkCwTRVrn32PXe9P6ZCA=='}


Now we encrypt the plaintext and PUT the encrypted data. Notice that there are no SSE headers in the response.

In [43]:
# Encrypt the downloaded data
cipher = AES.new(cse_key, AES.MODE_GCM)
ciphertext, tag = cipher.encrypt_and_digest(source['Body'].read())

encrypted_data = cipher.nonce + ciphertext + tag

# Upload the encrypted data
response = client.put_object(
    Bucket=DEMO_BUCKET_NAME,
    Key=FILENAME,
    Body=encrypted_data,
)
print_encryption_headers(response)

{}


Once more, delete the original!

In [44]:
response = client.delete_object(
    Bucket=DEMO_BUCKET_NAME,
    Key=FILENAME,
    VersionId=source['VersionId']
)
pprint(response['ResponseMetadata']['HTTPStatusCode'])

204


## Updating Default Encryption Settings

You can set default encryption on an existing bucket. The 200 response code indicates success.

In [45]:
response = client.put_bucket_encryption(
    Bucket=DEMO_BUCKET_NAME,
    ServerSideEncryptionConfiguration={
        'Rules': [
            {
                'ApplyServerSideEncryptionByDefault': {
                    'SSEAlgorithm': 'AES256'
                }
            },
        ]
    }
)
pprint(response['ResponseMetadata']['HTTPStatusCode'])

200
